In [1]:
import cv2
from ultralytics import YOLO
import numpy as np
import time
from collections import defaultdict
import subprocess
import torch

### Test Pytorch GPU


In [2]:
# --- Khởi tạo ---
print("Loading model...")
MODEL_NAME = "yolo11m.pt"

try:
    model = YOLO(MODEL_NAME).to("cuda")
    print("Model loaded on GPU (CUDA).")
except Exception as e:
    print(f"ERROR loading model on GPU: {e}")
    print(
        "Ensure PyTorch with CUDA support is installed correctly and CUDA drivers are up to date."
    )
    print("Falling back to CPU.")
    model = YOLO(MODEL_NAME)  # Tải trên CPU nếu có lỗi

Loading model...
Model loaded on GPU (CUDA).


- "1": "bicycle",
- "2": "car",
- "3": "motorcycle",
- "5": "bus",
- "7": "truck",


### 1. Static video


In [5]:
def count_vehicle(video_path, model_name, counting_line):
    # --- Cấu hình mặc định ---
    TARGET_CLASS_IDS = [1, 2, 3, 5, 7]
    CONFIDENCE_THRESHOLD = 0.5
    LINE_THICKNESS = 2
    FONT_SCALE = 1
    FONT_THICKNESS = 2
    FRAME_SKIP = 3
    FORGET_THRESHOLD_FRAMES = 150
    TOLERANCE = 1000
    LINE_P1 = tuple(counting_line[0])
    LINE_P2 = tuple(counting_line[1])

    def get_point_side(point, line_p1, line_p2):
        x, y = point
        x1, y1 = line_p1
        x2, y2 = line_p2
        cross_product = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
        if abs(cross_product) < TOLERANCE:
            return 0
        return 1 if cross_product > 0 else -1

    # --- Khởi tạo ---
    try:
        model = YOLO(model_name).to("cuda")
        print("Model loaded on GPU (CUDA).")
    except Exception as e:
        print(f"ERROR loading model on GPU: {e}")
        print(
            "Ensure PyTorch with CUDA support is installed correctly and CUDA drivers are up to date."
        )
        print("Falling back to CPU.")
        model = YOLO(model_name)  # Tải trên CPU nếu có lỗi

    # Lấy tên class mục tiêu
    class_names = model.names
    target_class_names = {cid: class_names[cid] for cid in TARGET_CLASS_IDS}
    print(f"Target class names: {target_class_names}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Lỗi: Không thể mở video {video_path}")
        return

    # Sử dụng defaultdict để dễ dàng khởi tạo bộ đếm và set ID
    # Lưu trữ số lượng đếm cho mỗi class name
    class_counts = defaultdict(int)
    # Lưu trữ các ID đã được đếm cho mỗi class name
    counted_ids_by_class = defaultdict(set)

    last_known_side = {}
    last_seen_frame = {}
    count = 0
    frame_count = 0
    start_time = time.time()
    processed_frame_count = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        if frame_count % FRAME_SKIP != 0:
            continue

        processed_frame_count += 1
        current_frame_tracks = set()

        results = model.track(
            frame,
            persist=True,
            verbose=False,
            conf=CONFIDENCE_THRESHOLD,
            classes=[TARGET_CLASS_IDS],
        )

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            # Lấy class ID cho mỗi box
            class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

            for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
                current_frame_tracks.add(track_id)
                last_seen_frame[track_id] = frame_count

                x1, y1, x2, y2 = box
                center_point = ((x1 + x2) // 2, (y1 + y2) // 2)

                current_side = get_point_side(center_point, LINE_P1, LINE_P2)
                prev_side = last_known_side.get(track_id)

                # Lấy tên class
                cls_name = class_names.get(cls_id, f"ID_{cls_id}")

                if current_side != 0:
                    if prev_side is not None and prev_side != current_side:
                        if track_id not in counted_ids_by_class[cls_name]:
                            class_counts[cls_name] += 1
                            counted_ids_by_class[cls_name].add(track_id)
                            print(
                                f"Frame {frame_count}: Counted {cls_name} (ID {track_id}). "
                                f"Total {cls_name}s: {class_counts[cls_name]}"
                            )

                    last_known_side[track_id] = current_side

                # --- Vẽ lên frame ---
                # Màu đỏ đối với các ID đã được đếm, màu xanh lá cho các ID chưa được đếm
                color = (
                    (0, 255, 0)
                    if track_id in counted_ids_by_class[cls_name]
                    else (0, 0, 255)
                )
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, LINE_THICKNESS // 2)
                label = f"{cls_name} ID:{track_id}"
                cv2.putText(
                    frame,
                    label,
                    (x1, y1 - 10),  # Vị trí text phía trên box
                    cv2.FONT_HERSHEY_SIMPLEX,
                    FONT_SCALE * 0.7,  # Font nhỏ hơn chút
                    color,
                    FONT_THICKNESS,
                )
                cv2.circle(frame, center_point, 5, (255, 0, 255), -1)

        # --- Xóa các track ID cũ không còn xuất hiện ---
        inactive_ids = []
        for track_id in list(last_seen_frame.keys()):
            if frame_count - last_seen_frame[track_id] > FORGET_THRESHOLD_FRAMES:
                inactive_ids.append(track_id)

        for track_id in inactive_ids:
            if track_id in last_seen_frame:
                del last_seen_frame[track_id]
            if track_id in last_known_side:
                del last_known_side[track_id]

        # Hiển thị số lượng đếm cho từng class
        display_y = 60  # Vị trí bắt đầu hiển thị text đếm
        total_count = 0
        for (
            cls_id
        ) in TARGET_CLASS_IDS:  # Duyệt theo thứ tự ID để ổn định vị trí hiển thị
            cls_name = class_names.get(cls_id)
            if cls_name:  # Chỉ hiển thị nếu class name tồn tại
                count = class_counts[cls_name]
                total_count += count
                text = f"{cls_name}: {count}"
                cv2.putText(
                    frame,
                    text,
                    (30, display_y),  # Vị trí hiển thị
                    cv2.FONT_HERSHEY_SIMPLEX,
                    FONT_SCALE,  # Kích thước font
                    (0, 255, 0),  # Màu xanh lá cây
                    FONT_THICKNESS + 1,  # Độ dày
                )
                display_y += 40  # Di chuyển xuống cho dòng text tiếp theo

        # Hiển thị tổng số lượng (tùy chọn)
        cv2.line(frame, LINE_P1, LINE_P2, (255, 0, 0), LINE_THICKNESS)
        cv2.putText(
            frame,
            f"Total: {total_count}",
            (30, display_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            FONT_SCALE,
            (255, 255, 0),
            FONT_THICKNESS + 1,
        )

        elapsed_time = time.time() - start_time
        if elapsed_time > 0:
            fps = processed_frame_count / elapsed_time
            cv2.putText(
                frame,
                f"FPS: {fps:.2f}",
                (frame.shape[1] - 250, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                FONT_SCALE,
                (0, 255, 0),
                FONT_THICKNESS,
            )

        scaled_frame = cv2.resize(frame, None, fx=0.8, fy=0.8)
        cv2.imshow("Suitcase Counter", scaled_frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

    # In kết quả cuối cùng ra console
    print("\n--- Final Counts ---")
    total_final_count = 0
    for cls_id in TARGET_CLASS_IDS:
        cls_name = class_names.get(cls_id)
        if cls_name:
            count = class_counts[cls_name]
            print(f"{cls_name}: {count}")
            total_final_count += count

    return dict(class_counts)

In [4]:
count_vehicle(
    video_path="input.mp4",
    model_name="yolo11m.pt",
    counting_line=np.array([[679, 604], [1271, 533]], dtype=np.int32),
)

Model loaded on GPU (CUDA).
Target class names: {1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
Frame 132: Counted motorcycle (ID 16). Total motorcycles: 1
Frame 135: Counted motorcycle (ID 6). Total motorcycles: 2
Frame 147: Counted motorcycle (ID 11). Total motorcycles: 3
Frame 177: Counted motorcycle (ID 26). Total motorcycles: 4
Frame 183: Counted motorcycle (ID 19). Total motorcycles: 5
Frame 228: Counted motorcycle (ID 32). Total motorcycles: 6
Frame 231: Counted motorcycle (ID 27). Total motorcycles: 7
Frame 234: Counted motorcycle (ID 18). Total motorcycles: 8
Frame 270: Counted motorcycle (ID 35). Total motorcycles: 9
Frame 378: Counted car (ID 4). Total cars: 1
Frame 453: Counted car (ID 7). Total cars: 2
Frame 492: Counted car (ID 28). Total cars: 3
Frame 546: Counted car (ID 33). Total cars: 4
Frame 555: Counted truck (ID 41). Total trucks: 1
Frame 639: Counted car (ID 42). Total cars: 5
Frame 678: Counted truck (ID 47). Total trucks: 2
Frame 702: Counted ca

{'bicycle': 0, 'car': 43, 'motorcycle': 16, 'bus': 0, 'truck': 9}

### 2. Live video


In [9]:
import yt_dlp

In [6]:
def get_youtube_stream_url(youtube_url, format_code="best[protocol=m3u8_native]"):
    """
    Lấy URL stream trực tiếp (HLS/m3u8) từ URL YouTube sử dụng yt-dlp.
    """
    ydl_opts = {
        "format": format_code,  # Ưu tiên lấy stream HLS (.m3u8)
        "quiet": True,  # Ít log hơn
        "noplaylist": True,  # Chỉ xử lý video đơn lẻ, không phải playlist
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(youtube_url, download=False)
            # Tìm URL stream trong kết quả
            stream_url = None
            if "url" in info_dict:
                stream_url = info_dict["url"]
            elif "formats" in info_dict:  # Đôi khi nằm trong danh sách formats
                for fmt in info_dict["formats"][::-1]:  # Duyệt từ chất lượng cao xuống
                    # Ưu tiên m3u8 nếu có, hoặc http khác nếu cần
                    if fmt.get("protocol") in ["m3u8_native", "m3u8"]:
                        stream_url = fmt.get("url")
                        print(
                            f"Found HLS stream URL: {stream_url[:50]}..."
                        )  # In ra phần đầu URL
                        break
                if (
                    not stream_url and info_dict["formats"]
                ):  # Lấy URL đầu tiên nếu không tìm thấy m3u8
                    stream_url = info_dict["formats"][0].get("url")
                    print(
                        f"Falling back to first available stream URL: {stream_url[:50]}..."
                    )

            if stream_url:
                print(f"Successfully extracted stream URL for {youtube_url}")
                return stream_url
            else:
                print(
                    f"ERROR: Could not extract stream URL for {youtube_url}. Check if the stream is live and public."
                )
                return None
    except yt_dlp.utils.DownloadError as e:
        print(f"ERROR using yt-dlp for {youtube_url}: {e}")
        print(
            "The video might be private, unavailable, region-locked, or the stream might have ended."
        )
        return None
    except Exception as e:
        print(f"An unexpected error occurred while getting stream URL: {e}")
        return None

In [7]:
def open_stream_with_processing(stream_url, width=1280, height=720):
    command = [
        "ffmpeg",
        "-i",
        stream_url,
        "-loglevel",
        "quiet",
        "-an",
        "-f",
        "rawvideo",
        "-pix_fmt",
        "bgr24",
        "-vf",
        f"scale={width}:{height}",
        "-",
    ]

    pipe = subprocess.Popen(command, stdout=subprocess.PIPE, bufsize=10**8)

    while True:
        raw_image = pipe.stdout.read(width * height * 3)
        if not raw_image:
            print("Stream ended or error.")
            break

        frame = (
            np.frombuffer(raw_image, dtype=np.uint8).reshape((height, width, 3)).copy()
        )

        cv2.putText(
            frame,
            "Live Stream",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 255, 0),
            2,
        )

        cv2.imshow("Processed Stream", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    pipe.terminate()
    cv2.destroyAllWindows()

In [20]:
stream_url = get_youtube_stream_url("https://www.youtube.com/watch?v=muijHPW82vI")

Successfully extracted stream URL for https://www.youtube.com/watch?v=muijHPW82vI


### 2.1 Phát live stream


In [11]:
open_stream_with_processing(stream_url, width=1280, height=720)

### 2.2. Đếm xe từ livestream


In [ ]:
import cv2
import time
import subprocess
import numpy as np
from ultralytics import YOLO
from collections import (
    defaultdict,
    deque,
)  # Use deque for efficient fixed-size queue if needed
import threading
import queue  # Use standard queue for thread safety


def count_vehicle_from_stream_optimized(
    stream_url,
    model_name,
    counting_line,
    display_width=1280,
    display_height=720,
    inference_size=640,
):
    # --- Configuration ---
    TARGET_CLASS_IDS = [
        1,2,3,5,7,
    ]  # bicycle, car, bus, truck (Adjust based on your model/needs)
    CONFIDENCE_THRESHOLD = 0.3
    LINE_THICKNESS = 2
    FONT_SCALE = 0.7  # Smaller font for potentially more info
    FONT_THICKNESS = 2
    FRAME_SKIP = 1  # Process every N frames (adjust based on performance)
    FORGET_THRESHOLD_FRAMES = 150  # Reduce if objects disappear quickly
    TOLERANCE = 10  # Fine-tune tolerance for crossing line detection
    LINE_P1 = tuple(counting_line[0])
    LINE_P2 = tuple(counting_line[1])
    QUEUE_MAX_SIZE = 5  # Max frames in queue to buffer

    # --- Utility Function ---
    def get_point_side(point, line_p1, line_p2):
        # Simplified cross-product calculation
        x, y = point
        x1, y1 = line_p1
        x2, y2 = line_p2
        val = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
        # Check magnitude against tolerance
        if abs(val) < TOLERANCE * np.linalg.norm(
            np.array(line_p2) - np.array(line_p1)
        ):  # Normalize tolerance by line length
            return 0  # On the line
        return 1 if val > 0 else -1

    # --- Load model ---
    try:
        model = YOLO(model_name).to("cuda")
        print("Model loaded on GPU (CUDA).")
        # Try using half precision for potential speedup on compatible GPUs
        try:
            # model.half()
            print("Model set to half precision (FP16).")
        except Exception as e:
            print(f"Could not set model to half precision: {e}")
    except Exception as e_cuda:
        print(f"ERROR loading model on GPU: {e_cuda}. Falling back to CPU.")
        model = YOLO(model_name)  # Load on CPU

    class_names = model.names
    print("Model class names:", class_names)
    valid_target_ids = [cid for cid in TARGET_CLASS_IDS if cid in class_names]
    if len(valid_target_ids) != len(TARGET_CLASS_IDS):
        print("Warning: Some TARGET_CLASS_IDS not found in model.")
    TARGET_CLASS_IDS = valid_target_ids
    print(
        f"Counting target classes: { {cid: class_names[cid] for cid in TARGET_CLASS_IDS} }"
    )

    # --- Thread-Safe Data Structures ---
    frame_queue = queue.Queue(maxsize=QUEUE_MAX_SIZE)
    stop_signal = threading.Event()

    # --- Frame Reader Thread ---
    def frame_reader_thread(url, q, stop_event, w, h):
        print("Reader thread started.")
        command = [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",  # Quieter ffmpeg logs
            "-i",
            url,
            "-vf",
            f"fps=30,scale={w}:{h}",  # Optional: Limit input FPS, Scale input
            "-pix_fmt",
            "bgr24",  # Pixel format OpenCV understands
            "-preset",
            "ultrafast",  # For faster encoding/piping (might reduce quality slightly)
            "-tune",
            "zerolatency",  # Reduce latency
            "-f",
            "rawvideo",  # Output raw video frames
            "-an",  # No audio
            "-",  # Output to stdout
        ]
        pipe = None
        read_count = 0
        try:
            pipe = subprocess.Popen(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                bufsize=w * h * 3,
            )  # Adjusted buffer
            frame_size = w * h * 3
            while not stop_event.is_set():
                raw_image = pipe.stdout.read(frame_size)
                if not raw_image or len(raw_image) != frame_size:
                    print("Reader thread: End of stream or read error.")
                    break

                read_count += 1
                if read_count % FRAME_SKIP == 0:
                    frame = (
                        np.frombuffer(raw_image, dtype=np.uint8)
                        .reshape((h, w, 3))
                        .copy()
                    )
                    try:
                        # Put frame in queue, wait briefly if full
                        q.put(frame, timeout=0.5)
                    except queue.Full:
                        # If queue is still full after waiting, drop the oldest frame and add the new one
                        try:
                            q.get_nowait()
                            q.put(frame, block=False)  # Put without blocking now
                            # print("Reader thread: Queue full, dropped oldest frame.")
                        except queue.Empty:
                            pass  # Should not happen if it was Full
                        except queue.Full:
                            pass  # Failed to add even after dropping (very high load)
            pipe.stdout.flush()
            pipe.stderr.flush()
        except Exception as e:
            print(f"Reader thread exception: {e}")
        finally:
            if pipe and pipe.poll() is None:
                print("Reader thread: Terminating ffmpeg process...")
                pipe.terminate()
                try:
                    pipe.wait(timeout=2)  # Wait for ffmpeg to exit
                except subprocess.TimeoutExpired:
                    print(
                        "Reader thread: ffmpeg did not terminate gracefully, killing."
                    )
                    pipe.kill()
            print("Reader thread finished.")
            stop_event.set()  # Signal main thread to stop if reader stops

    # --- Main Processing Logic ---
    reader = threading.Thread(
        target=frame_reader_thread,
        args=(stream_url, frame_queue, stop_signal, display_width, display_height),
        daemon=True,
    )
    reader.start()

    class_counts = defaultdict(int)
    counted_ids_by_class = defaultdict(set)
    processed_frame_count = 0
    side_history = defaultdict(lambda: deque(maxlen=20))
    start_time = time.time()
    display_fps = 0
    last_fps_update_time = start_time
    fps_update_interval = 1.0  # Update FPS display every second
    last_seen_frame = {}

    while reader.is_alive() or not frame_queue.empty():
        try:
            # Get frame from queue, wait up to 1 second
            display_frame = frame_queue.get(timeout=1.0)
        except queue.Empty:
            if not reader.is_alive():
                print("Main loop: Reader stopped and queue empty. Exiting.")
                break
            else:
                print("Main loop: Queue empty, reader still alive. Waiting...")
                continue

        processed_frame_count += 1

        # --- Prepare frame for inference ---
        # Resize frame for faster inference
        inference_frame = cv2.resize(
            display_frame,
            (inference_size, int(display_height * inference_size / display_width)),
        )

        scale_w = inference_frame.shape[1] / display_width
        scale_h = inference_frame.shape[0] / display_height

        # --- Run Inference and Tracking ---
        results = model.track(
            inference_frame,  # Use resized frame
            persist=True,
            verbose=False,
            conf=CONFIDENCE_THRESHOLD,
            classes=TARGET_CLASS_IDS,
            device="cuda" if torch.cuda.is_available() else "cpu",  # Explicit device
        )

        # --- Process Detections ---
        current_frame_tracks = set()
        if results[0].boxes is not None and results[0].boxes.id is not None:
            # Get data and transfer to CPU efficiently
            boxes_inf = (
                results[0].boxes.xyxy.cpu().numpy()
            )  # Boxes in inference coordinates
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            class_ids = results[0].boxes.cls.cpu().numpy().astype(int)

            for box_inf, track_id, cls_id in zip(boxes_inf, track_ids, class_ids):
                current_frame_tracks.add(track_id)
                last_seen_frame[track_id] = processed_frame_count  # Use processed count

                # --- Scale coordinates back to display frame size ---
                x1_inf, y1_inf, x2_inf, y2_inf = box_inf
                x1 = int(x1_inf / scale_w)
                y1 = int(y1_inf / scale_h)
                x2 = int(x2_inf / scale_w)
                y2 = int(y2_inf / scale_h)

                # Ensure coordinates are within display frame bounds
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(display_width - 1, x2), min(display_height - 1, y2)

                # Use bottom center point for crossing detection (often more stable for vehicles)
                center_point = ((x1 + x2) // 2, (y1 + y2) // 2)

                current_side = get_point_side(center_point, LINE_P1, LINE_P2)
                side_history[track_id].append(current_side)
                cls_name = class_names.get(cls_id, f"ID_{cls_id}")

                if (
                    not side_history[track_id]
                    or side_history[track_id][-1] != current_side
                ):
                    side_history[track_id].append(current_side)

                # if track_id != 1:
                #     print(f"[HISTORY] Counted {cls_name} (ID {track_id}. History: {list(side_history[track_id])}. Total {cls_name}s: {class_counts[cls_name]}")
                if track_id not in counted_ids_by_class[cls_name]:
                    history = side_history[track_id]
                    # Kiểm tra xem trong lịch sử (deque) có cả 1 và -1 không
                    has_side1 = 1 in history
                    has_side_minus1 = -1 in history

                    if has_side1 and has_side_minus1:
                        # Đã đi qua cả hai bên -> Rất có khả năng đã vượt qua vạch
                        class_counts[cls_name] += 1
                        counted_ids_by_class[cls_name].add(track_id)
                        print(
                            f"[HISTORY] Counted {cls_name} (ID {track_id}. History: {list(history)}. Total {cls_name}s: {class_counts[cls_name]}"
                        )

                # --- Draw on Display Frame ---
                color = (
                    (0, 255, 0)
                    if track_id in counted_ids_by_class[cls_name]
                    else (0, 0, 255)
                )
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, LINE_THICKNESS)
                label = f"{cls_name}:{track_id}"  # Shorter label
                cv2.putText(
                    display_frame,
                    label,
                    (x1, y1 - 5),  # Adjusted position
                    cv2.FONT_HERSHEY_SIMPLEX,
                    FONT_SCALE * 0.8,
                    color,
                    FONT_THICKNESS,
                )
                cv2.circle(
                    display_frame, center_point, 3, (255, 0, 255), -1
                )  # Smaller circle

        # --- Forget Old Track IDs ---
        inactive_ids = [
            tid
            for tid, last_frame in list(last_seen_frame.items())
            if processed_frame_count - last_frame > FORGET_THRESHOLD_FRAMES
        ]
        for tid in inactive_ids:
            last_seen_frame.pop(tid, None)
            side_history.pop(tid, None)  # Remove history for inactive IDs

        # --- Display Information ---
        cv2.line(
            display_frame, LINE_P1, LINE_P2, (255, 0, 0), LINE_THICKNESS + 1
        )  # Slightly thicker line
        y_text = 40
        total_count = 0
        # Sort by class ID for consistent display order
        for cid in sorted(
            [id for id in TARGET_CLASS_IDS if class_names.get(id) in class_counts]
        ):
            cname = class_names.get(cid)
            count = class_counts[cname]
            total_count += count
            cv2.putText(
                display_frame,
                f"{cname}: {count}",
                (20, y_text),
                cv2.FONT_HERSHEY_SIMPLEX,
                FONT_SCALE,
                (0, 255, 255),
                FONT_THICKNESS,
            )  # Yellow text
            y_text += 25  # Smaller gap

        # --- Calculate and Display FPS ---
        current_time = time.time()
        if current_time - last_fps_update_time >= fps_update_interval:
            elapsed_time = current_time - start_time
            display_fps = (
                processed_frame_count / elapsed_time if elapsed_time > 0 else 0
            )
            last_fps_update_time = current_time  # Reset update timer

        cv2.putText(
            display_frame,
            f"FPS: {display_fps:.1f}",
            (display_width - 150, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            FONT_SCALE,
            (0, 255, 0),
            FONT_THICKNESS,
        )  # Green FPS

        # --- Show Frame ---
        cv2.imshow("Live Vehicle Count (Optimized)", display_frame)

        # --- Exit Condition ---
        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("Main loop: Quit key pressed.")
            stop_signal.set()  # Signal reader thread to stop
            break

    # --- Cleanup ---
    print("Main loop: Cleaning up...")
    stop_signal.set()  # Ensure signal is set
    if reader.is_alive():
        print("Main loop: Waiting for reader thread to finish...")
        reader.join(timeout=5.0)  # Wait for reader thread with timeout
        if reader.is_alive():
            print("Main loop: Reader thread did not exit gracefully.")

    cv2.destroyAllWindows()

    # --- Final Results ---
    print("\n--- Final Counts ---")
    total_final = 0
    # Sort final output alphabetically by class name
    for cname in sorted(class_counts.keys()):
        count = class_counts[cname]
        print(f"{cname}: {count}")
        total_final += count
    print(f"Total Vehicles Counted: {total_final}")
    print("--------------------\n")

    return dict(class_counts)

In [65]:
model_path = "yolo11m.pt"

display_w, display_h = 1280, 720
infer_sz = 640

# Vạch kẻ ngang để đếm xe
line_pt1 = (8, 400)
line_pt2 = (800, 250)
counting_line_coords = [line_pt1, line_pt2]

print(f"Starting optimized counting on stream: {stream_url}")

final_counts = count_vehicle_from_stream_optimized(
    stream_url,
    model_path,
    counting_line_coords,
    display_width=display_w,
    display_height=display_h,
    inference_size=1280,
)

print("\nFunction returned final counts:")
print(final_counts)

Starting optimized counting on stream: https://manifest.googlevideo.com/api/manifest/hls_playlist/expire/1745265901/ei/jVAGaP21LcuRvcAP84_78Ag/ip/2405:4803:d3d0:e1b0:1508:df0a:4e8e:83d7/id/muijHPW82vI.3/itag/96/source/yt_live_broadcast/requiressl/yes/ratebypass/yes/live/1/sgoap/gir%3Dyes%3Bitag%3D140/sgovp/gir%3Dyes%3Bitag%3D137/rqh/1/hls_chunk_host/rr8---sn-42u-q5qs.googlevideo.com/xpc/EgVo2aDSNQ%3D%3D/playlist_duration/30/manifest_duration/30/bui/AccgBcM5WNAohrqEhe2QgrH8XSWAo5OhnP0-21KtDBie0O4BUta3nQ9k3WU0Rtj0HpphTWjFjquo4gvN/spc/_S3wKpRMWGqJqfrtl0lT7mhif4sRpST_-rmTPEUnUs6SKmOYw1F2Jfy66XpkeUnwG16Of_Q/vprv/1/playlist_type/DVR/initcwndbps/2741250/met/1745244303,/mh/EU/mm/44/mn/sn-42u-q5qs/ms/lva/mv/m/mvi/8/pl/46/rms/lva,lva/dover/11/pacing/0/keepalive/yes/fexp/51355912/mt/1745244035/sparams/expire,ei,ip,id,itag,source,requiressl,ratebypass,live,sgoap,sgovp,rqh,xpc,playlist_duration,manifest_duration,bui,spc,vprv,playlist_type/sig/AJfQdSswRAIgegCs5OlhJLzCOZoTvOCXgqO0fjhowv-AK4hmBfNjMyIC

### Link DEMO

- Static video: https://drive.google.com/file/d/1BUr82BHHbs3yZg2I4LP3nZBHa3Kj51fj/view?usp=drive_link
- Live video: https://drive.google.com/file/d/1qVRgvENXwPg3yMCOQfC8jHYP8Wk9TW6_/view?usp=drive_link